In [1]:
# Loading the dataset
import pandas as pd

df: pd.DataFrame = pd.read_csv("data/stock prices.csv", usecols=["date", "close", "high", "low", "open", "adjClose", "adjHigh", "adjLow", "adjOpen"]).dropna()

# Sorting based on date column
df = df.sort_values(by="date").set_index(keys="date", drop=True)
df.head()

,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
date,,,,,,,,
2015-05-27 00:00:00+00:00,132.045,132.260,130.05,130.34,121.682558,121.880685,119.844118,120.111360
2015-05-28 00:00:00+00:00,131.780,131.950,131.10,131.86,121.438354,121.595013,120.811718,121.512076
2015-05-29 00:00:00+00:00,130.280,131.450,129.90,131.23,120.056069,121.134251,119.705890,120.931516
2015-06-01 00:00:00+00:00,130.535,131.390,130.05,131.20,120.291057,121.078960,119.844118,120.903870
2015-06-02 00:00:00+00:00,129.960,130.655,129.32,129.86,119.761181,120.401640,119.171406,119.669029


In [3]:
# Splitting the dataset sequentially
test_split = 0.2
train_df: pd.Series = df.iloc[ : int(df.shape[0] * (1 - test_split)), : ]
test_df: pd.Series = df.iloc[int(df.shape[0] * (1 - test_split)): , : ]

X_train: pd.DataFrame = train_df.drop(columns=['close'])
y_train: pd.Series = train_df['close']

X_test: pd.DataFrame = test_df.drop(columns=['close'])
y_test: pd.Series = test_df['close']

# Shapes
print(X_train.shape)
print(X_test.shape)

(1006, 7)
(252, 7)


In [4]:
# Scaling the input features
from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler() # This is for all features
scaler_x.fit(X_train)

scaler_y = StandardScaler() # This is just for labels
scaler_y.fit(y_train.to_frame())

# Transforming train and test input features —→ NumPy Array
X_train = scaler_x.transform(X_train)
X_test = scaler_x.transform(X_test)

y_train = scaler_y.transform(y_train.to_frame())
y_test = scaler_y.transform(y_test.to_frame())

# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(1006, 7)
(252, 7)
(1006, 1)
(252, 1)


In [5]:
# Dataset Preperation for stock price predictor model
import numpy as np

def prepare_dataset(dataset: np.array, timestep: int = 1) -> pd.DataFrame:
    dataset = pd.DataFrame(dataset)
    periods = list(range(1, timestep + 1))
    return dataset.shift(periods).iloc[timestep + 1:].to_numpy()

# timestep —→ Hyperparameter (More would be great)
timestep = 50
X_train = prepare_dataset(dataset=X_train, timestep=timestep)
X_test = prepare_dataset(dataset=X_test, timestep=timestep)

y_train = y_train[timestep + 1 : ]
y_test = y_test[timestep + 1 : ]

# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(955, 350)
(201, 350)
(955, 1)
(201, 1)


In [6]:
# Model Building
import torch
from torch.nn import Module, RNN, Linear
from typing import Literal

device: Literal['cpu', 'cuda'] = 'cuda' if torch.cuda.is_available() else 'cpu'

class CustomModel(Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.rnn = RNN(input_size=input_size, hidden_size=25, num_layers=1, nonlinearity="tanh", batch_first=True, bidirectional=False)
        self.linear = Linear(in_features=25, out_features=1)
        
    def forward(self, X_train):
        _, final = self.rnn(X_train)
        y_pred = self.linear(final)
        return y_pred

In [ ]:
# Training parameters
learning_rate: float = 0.001
epochs: int = 2500

# Loss function and optimizer
model: CustomModel = CustomModel(input_size=X_train.shape[1] // timestep).to(device)
model.train()

criterion = torch.nn.modules.loss.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(params = model.parameters())

In [8]:
# Training pipeline
for i in range(epochs):

    # Converting data from NumPy array to Tensor
    X_train_cp: torch.Tensor = torch.tensor(X_train, dtype=torch.float32).reshape(X_train.shape[0], timestep, X_train.shape[1] // timestep).to(device) # rnn compatable reshaping —→ (batch_size, timesteps, embeddings)
    y_train_cp: torch.Tensor = torch.tensor(y_train, dtype=torch.float32).to(device)

    # Initializing all gradients to zero
    optimizer.zero_grad()

    # Forward propogation
    y_pred = model(X_train_cp)

    # Loss calculation
    loss = criterion(y_pred.reshape(-1, 1), y_train_cp.reshape(-1, 1)) # Loss requires shape (batch_size, 1[prediction])

    # Gradient calculation
    loss.backward()

    # Updating weights
    optimizer.step()

    if (i + 1) % 100 == 0:
        print(f"Epoch {i + 1} → Loss = {loss.item()}")

Epoch 100 → Loss = 0.08576169610023499
Epoch 200 → Loss = 0.08282734453678131
Epoch 300 → Loss = 0.070900097489357
Epoch 400 → Loss = 0.03813648223876953
Epoch 500 → Loss = 0.036644525825977325
Epoch 600 → Loss = 0.018236039206385612
Epoch 700 → Loss = 0.025697194039821625
Epoch 800 → Loss = 0.0224065650254488
Epoch 900 → Loss = 0.01382372248917818
Epoch 1000 → Loss = 0.012014418840408325
Epoch 1100 → Loss = 0.01155745517462492
Epoch 1200 → Loss = 0.009920327924191952
Epoch 1300 → Loss = 0.008782916702330112
Epoch 1400 → Loss = 0.007605293300002813
Epoch 1500 → Loss = 0.006777410861104727
Epoch 1600 → Loss = 0.006312579847872257
Epoch 1700 → Loss = 0.00637750094756484
Epoch 1800 → Loss = 0.005949118174612522
Epoch 1900 → Loss = 0.005153784062713385
Epoch 2000 → Loss = 0.005015408154577017
Epoch 2100 → Loss = 0.005027846898883581
Epoch 2200 → Loss = 0.005932950880378485
Epoch 2300 → Loss = 0.0046887085773050785
Epoch 2400 → Loss = 0.004673060029745102
Epoch 2500 → Loss = 0.0041312491521

In [9]:
model.eval()

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [ ]:
with torch.no_grad():
    # Conterting data from NumPy array to Tensor
    X_test_cp: torch.Tensor = torch.tensor(X_test, dtype=torch.float32).reshape(X_test.shape[0], timestep, X_test.shape[1] // timestep).to('cuda')
    y_test_cp: torch.Tensor = torch.tensor(y_test, dtype=torch.float32).to('cuda')

    # Forward propogation
    y_pred = model(X_test_cp).to('cpu')

In [11]:
import plotly.graph_objects as go
import numpy as np

# Ensure data is 1D arrays for Plotly
# If y_test or y_pred are 2D (e.g., shape (n_samples, 1)), flatten them
actual_prices = scaler_y.inverse_transform(y_test).flatten()
predicted_prices = scaler_y.inverse_transform(y_pred.reshape(1, -1)).flatten()

# Create the x-axis (Days). 
# Assuming len(actual_prices) represents the number of days.
days = np.arange(len(actual_prices))

# Create the figure
fig = go.Figure()

# Add Actual Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=actual_prices,
    mode='lines',
    name='Actual Stock Prices',
    line=dict(color='blue', width=2)
))

# Add Predicted Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices,
    mode='lines',
    name='Predicted Stock Prices',
    line=dict(color='orange', width=2, dash='dash')
))

# Update layout
fig.update_layout(
    title='Stock Price Prediction',
    xaxis_title='Days',
    yaxis_title='Price',
    hovermode='x unified',
    template='plotly_white',
    width=1000,  # Adjust width to match your matplotlib figsize preference
    height=500
)

# Show the chart
fig.show()